# Requesting 30 Instruments Historical Data Synchronously using Data Library get_history

Importing requires libraries

In [1]:
import os
import time
from IPython.display import Markdown, display
import lseg.data as ld
from lseg.data import session
from lseg.data._errors import LDError
from dotenv import load_dotenv

Loading Platform session (Data Platform) credential from `.env` file.

In [2]:
# Load environment variables from .env file
load_dotenv(dotenv_path='.env')
# Retrieve Platform Session credentials from environment variables
app_key = os.getenv('LSEG_API_KEY')
username = os.getenv('LSEG_MACHINE_ID')
password = os.getenv('LSEG_PASSWORD')

Define required variables.

In [5]:
# -- Top 30 S&P 500 companies ----------------------------------------------------
INSTRUMENTS = [
"NVDA.O",
"AAPL.O",
"MSFT.O",
"AMZN.O",
"GOOG.O",
"AVGO.O",
"META.O",
"ORCL.N",
"IBM.N",
"PLTR.O",
"NFLX.O",
"TSLA.O",
"CRM.N",
"AMD.O",
"INTC.O",
"ARM.O",
"TXN.O",
"CSCO.O",
"WMT.O",
"LLY.N",
"JPM.N",
"XOM.N",
"V.N",
"JNJ.N",
"MU.O",
"MA.N",
"COST.O",
"CVX.N",
"BAC.N",
"CAT.N",
]

# ── Date range ─────────────────────────────────────────────────────────────────
START = "2025-11-01T00:00:00Z"
END   = "2026-02-28T23:59:59Z"

# ── Event correction filters ───────────────────────────────────────────────────
EVENT_ADJUSTMENTS = ["exchangeCorrection",
                    "manualCorrection",
                    "CCH",
                    "CRE",
                    "RPO",
                    "RTS"]

# ── Field lists ─────────────────────────────────────────────────────────────────
INTERDAY_FIELDS = ["BID", "ASK", "OPEN_PRC", "HIGH_1", "LOW_1", "TRDPRC_1", "NUM_MOVES", "TRNOVR_UNS"]

Initialize and open the session.

In [6]:
# Create a platform session with provided credentials for authentication
ld_session = session.platform.Definition(
         app_key=app_key,
         grant=session.platform.GrantPassword(
             username=username,
             password=password
         ),
         signon_control=True
).get_session()

# Set this session as the default for all subsequent Data Library calls
session.set_default(ld_session)

# Open the connection to the LSEG Data Platform
ld_session.open()

<OpenState.Opened: 'Opened'>

Requests 30 Instruments (single-RIC looping) synchronously with time counter.

In [8]:
hist_data = []
display(Markdown("**Start the wall-clock timer...**"))
start_time = time.perf_counter()
try:
    for instrument in INSTRUMENTS:
        # Fetch historical pricing data for the instrument
        data = ld.get_history(
            universe=instrument,
            start=START,
            end=END,
            adjustments=EVENT_ADJUSTMENTS,
            fields=INTERDAY_FIELDS,
            interval="1D"
        )
        hist_data.append(data)
        #time.sleep(1)  # Sleep for 1 second to avoid hitting rate limits
        
except LDError as e:
    print(f"**Exception:** {e}")
finally:
    elapsed = time.perf_counter() - start_time
    elapsed_string = f"**Sending Historical-Pricing Interday Historical Summaries (Daily) for ({len(INSTRUMENTS)}) RICs in {elapsed:0.2f} seconds.**"
    display(Markdown(elapsed_string))

**Start the wall-clock timer...**

**Sending Historical-Pricing Interday Historical Summaries (Daily) for (30) RICs in 2.82 seconds.**

Closing the session.

In [9]:
# Close the session to release resources; in a long-running application, consider keeping the session open and reusing it for subsequent API calls instead.
ld_session.close()
ld.close_session()